# 12 · Calibration, validation thresholds and immutable analysis lock

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Flagship only. All five anchor seeds need OOF selectors and independently adjudicated calibration/validation claims. Do not use this notebook to bypass missing data.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Confirm the registered analysis before test access

In [ ]:
from oncoplate.governance import require_gate
from oncoplate.pipeline import calibrate_predictor
from oncoplate.inputs import make_partition_candidates
from oncoplate.benchmark import adjudicated_ratings
from oncoplate.evaluation import fit_support_calibrators,selected_calibrators,score_candidates,policy_thresholds
require_gate(cfg,'domain_schema')
assert cfg['study']['dataset']!='foodnextdb','Use notebook 21 for the separate foundation analysis.'
RUN_IDS=[f'dinov2_vits14_finetune_joint_s{s}' for s in cfg['study']['model_seeds']]
context=read_json(p['private']/'protocol_context.json')
CALIBRATION_KIND='temperature'  # Registered anchor; sigmoid/identity are separate sensitivity results.
PRIMARY_COVERAGE=.8
print(RUN_IDS)

## 2. Fit predictor calibration on calibration labels
Predictor calibration is evaluated separately. Shared selector input features remain the same raw-predictor contract as OOF.

In [ ]:
for run_id in RUN_IDS:
    calibrators,metrics=calibrate_predictor(cfg,run_id)
    print(run_id,{k:v['log_loss'] for k,v in metrics.items()})

## 3. Fit support calibration and lock validation-derived thresholds

In [ ]:
for run_id in RUN_IDS:
    sel=p['private']/'selectors'/run_id
    claim_dir=p['private']/'claims'/run_id/context['input_mode']
    candidates={};ratings={}
    for partition in ['calibration','validation']:
        cc,_,_=make_partition_candidates(cfg,run_id,partition,input_mode=context['input_mode'],asof=context['asof'])
        candidates[partition]=cc
        ratings[partition]=adjudicated_ratings(read_table(claim_dir/f'{partition}_ratings_adjudicated.csv'))
    all_cals=fit_support_calibrators(candidates['calibration'],ratings['calibration'],sel,sel/'support_calibrators.json')
    chosen=selected_calibrators(all_cals,CALIBRATION_KIND)
    write_json(sel/'chosen_support_calibrators.json',chosen)
    val=score_candidates(candidates['validation'],sel,chosen)
    thresholds=policy_thresholds(val,sel/'operating_thresholds.json',mixture=cfg['study']['domain_mixture'],targets=cfg['study']['coverage_targets'])
    print(run_id, 'M7 validation coverage:',thresholds['M7'][str(PRIMARY_COVERAGE)]['validation_coverage'])

## 4. Bind code, configs, checkpoints, sources, schemas, thresholds and reference snapshot IDs
This is a procedural reproducibility lock, not access control against the data owner. A later change requires an explicit versioned amendment.

In [ ]:
from oncoplate.governance import make_lock
files=list((REPO/'src/oncoplate').glob('*.py'))+[REPO/'config/default.yaml',p['prepared']/'split_manifest.csv',p['private']/'claim_rules.json',p['private']/'inference_states.json',p['private']/'protocol_context.json']
files += [p['prepared']/f'targets_{h}'/'schema.json' for h in ['independent','joint']]
files += [p['prepared']/f'targets_{h}'/'targets.npz' for h in ['independent','joint']]
for run_id in RUN_IDS:
    run=p['runs']/run_id;sel=p['private']/'selectors'/run_id
    files += [run/'best.pt',run/'run.json',sel/'generic.json',sel/'pcsi.json',sel/'chosen_support_calibrators.json',sel/'operating_thresholds.json']
protocol={'run_ids':RUN_IDS,'primary_contrast':['M7','M3'],'target_coverage':PRIMARY_COVERAGE,'coverage_margin':.05,'question_budget':0,
          'domain_mixture':cfg['study']['domain_mixture'],'bootstrap_replicates':1000,'model_seeds':cfg['study']['model_seeds'],
          'calibration_kind':CALIBRATION_KIND,'input_mode':context['input_mode'],'asof':context['asof'],
          'external_protocol':'New external records must be evaluated under the separate cohort-addendum workflow in notebook 22; never modify locked main targets.'}
lock_path=p['private']/'analysis_lock.json'
lock=make_lock(lock_path,files,protocol)
print('Locked:',lock_path,lock['lock_hash'])

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
